In [1]:
from analytics.db import query

In [2]:
query("""
 -- Warm-up 5. For each payment type, how many different orders used it? What percent of all paid orders is that? Round to 1 decimal.
SELECT
    payment_type,
    COUNT(DISTINCT order_id) AS orders_count,
    ROUND(100.0 * COUNT(DISTINCT order_id) / (SELECT COUNT(DISTINCT order_id) FROM order_payments), 1) AS pct_of_total
FROM order_payments
GROUP BY payment_type
ORDER BY orders_count DESC
         """)

,payment_type,orders_count,pct_of_total
0,credit_card,76505,76.9
1,boleto,19784,19.9
2,voucher,3866,3.9
3,debit_card,1528,1.5
4,not_defined,3,0.0


In [3]:
from analytics.paths import PROJECT_ROOT

WORK = PROJECT_ROOT / "data" / "work"
WORK.mkdir(parents=True, exist_ok=True)

august = "order_purchase_timestamp >= '2018-08-01' AND order_purchase_timestamp < '2018-09-01'"

orders = query(f"""
       SELECT order_id, order_status, order_purchase_timestamp
       FROM orders
       WHERE {august}
   """)
items = query(f"""
       SELECT order_id, order_item_id, price, freight_value
       FROM order_items
       WHERE order_id IN (SELECT order_id FROM orders WHERE {august})
   """)
payments = query(f"""
       SELECT order_id, payment_sequential, payment_type, payment_value
       FROM order_payments
       WHERE order_id IN (SELECT order_id FROM orders WHERE {august})
   """)

orders.to_csv(WORK / "aug2018_orders.csv", index=False)
items.to_csv(WORK / "aug2018_items.csv", index=False)
payments.to_csv(WORK / "aug2018_payments.csv", index=False)

len(orders), len(items), len(payments)

(6512, 7248, 6698)

> **Data Extraction Note:** Extracted August 2018 data from the database using SQL filtering with `IN (SELECT ...)` to ensure data integrity, and exported the resulting rows into local CSV files inside `data/work/` for spreadsheet analysis.

## Review 4 — Bug 1: Revenue Definition and Check Formatting

I inspected Revenue!B4 and compared its SUMIFS sum range with the items
column headers and the Day 4 revenue definition.

B4 included freight, while the Day 4 definition includes item prices only,
excluding canceled and unavailable orders. The two cells therefore
measured different things.

I also inspected the conditional-formatting rules for D2:D4 and found
that a nonzero difference was not highlighted in red.

I used the supplied review answers and AI guidance to understand the
issue, then checked it directly in the workbook.

### Verification

With the original August date range, all three differences were zero
and highlighted in green.

Changing Inputs!B2 to 2018-08-02 made all three differences nonzero
and highlighted them in red. I restored Inputs!B2 to 2018-08-01 and
confirmed that the differences returned to zero.